# DeepRacer-Genesis on Colab (T4)

AWS DeepRacer RL environment on [Genesis](https://genesis-world.readthedocs.io/) —
GPU-batched physics, TorchRL PPO, config-as-code experiments.

This notebook trains a state-vector driving policy with the experiment
framework (~10-15 min total on a T4, most of it one-time JIT compilation) and
renders a bird's-eye video of the trained cars.

Colab note: Colab VMs have no NVIDIA graphics userland, so the Madrona/Nyx
*camera-observation* renderers can't run here — feature-vector training runs
entirely on CUDA, and videos render through Mesa (software, slower).

In [ ]:
# @title GPU check
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
# @title Install deepracer-genesis
REPO = "https://github.com/Luna-v0/deepracer-genesis.git"
BRANCH = "main"

# Colab preinstalls TensorFlow, whose bundled LLVM symbols clash with Mesa's
# software renderer (segfault at first render once tensorboard imports). We
# don't use TF -> remove it.
!pip uninstall -q -y tensorflow tensorflow-cpu tf-keras 2>/dev/null

# --force-reinstall --no-cache-dir: after a "Restart runtime" the VM disk (and
# any previously installed/broken build of this package) survives — never
# trust a cached install when switching branches.
import glob
local_wheel = sorted(glob.glob("/content/deepracer_genesis-*.whl"))
if local_wheel:
    print("installing local wheel:", local_wheel[-1])
    !pip install -q --force-reinstall --no-deps {local_wheel[-1]}
    !pip install -q {local_wheel[-1]}
else:
    !pip install -q --no-cache-dir --force-reinstall --no-deps "deepracer-genesis @ git+{REPO}@{BRANCH}"
    !pip install -q "deepracer-genesis @ git+{REPO}@{BRANCH}"
!pip install -q torchrl tensordict

# fail fast if the install is unusable
import subprocess, sys
r = subprocess.run([sys.executable, "-c",
    "from deepracer_genesis.experiment import Experiment, run; "
    "from deepracer_genesis import ASSETS_DIR; import os; "
    "assert os.path.exists(os.path.join(ASSETS_DIR, 'routes', 'reinvent_base.npy')), 'assets missing'; "
    "print('install OK: experiment framework + assets present')"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode != 0:
    raise RuntimeError("deepracer-genesis installed but is incomplete — "
                       "rerun this cell, or Restart runtime and rerun.")

## The experiment: one file, one class

Training config lives as class attributes; the env / DR / policy pipeline is
a `>>` chain. Variants are subclasses. The spec is content-hashed — rerunning
an identical config is a cache hit, and every run lands in
`runs/{group}/{variant}-{seed}-{id}/` with checkpoints, TensorBoard logs and
an `eval_record.json`.

In [ ]:
%%writefile /content/colab_experiment.py
"""Single-file experiment — the whole config surface in one class."""
from deepracer_genesis.experiment import (
    Experiment,
    FeatureEnvironment,
    VectorPolicy,
)


class ColabRacer(Experiment):
    # training configuration
    seed = 0
    total_env_steps = 5_000_000       # ~8 min at the T4's ~10k steps/s
    eval_every_steps = 1_000_000      # deterministic eval every 1M steps
    ablation_group = "colab"
    variant = "feature"

    # experiment hyperparameters (any name you like)
    num_envs = 512                    # T4-sized fleet
    lookahead_k = 10

    # the pipeline: env >> (optional DR stages) >> policy
    def pipeline(self):
        return (
            FeatureEnvironment(lookahead_k=self.lookahead_k,
                               num_envs=self.num_envs)
            >> VectorPolicy(keys=("state",))
        )


if __name__ == "__main__":
    record = ColabRacer().run(root="/content/runs")
    print("\nfinal eval:", {k: round(v, 3) for k, v in record.metrics.items()
                            if isinstance(v, (int, float))})
    print("eval history:", [(h["frames"], round(h["completion_rate"], 2))
                            for h in record.eval_history])

In [ ]:
# @title Train it (first run includes Genesis JIT compilation — a few minutes on a T4)
!cd /content && python colab_experiment.py

In [ ]:
%%writefile /content/make_video.py
# render the trained policy: bird's-eye spectator video, all cars at once
# (rollout_video strips domain randomization -> nominal-condition footage)
import colab_experiment  # noqa: F401  (registers ColabRacer)
from deepracer_genesis.experiment.visualize import rollout_video

path = rollout_video("ColabRacer", root="/content/runs",
                     steps=300, num_envs=8,
                     spectator_res=(640, 480))   # Mesa software rendering: keep it modest
print("VIDEO:", path)

In [ ]:
# @title Render the video (Mesa software rendering: ~35 s warmup, then ~0.4 s/frame)
!cd /content && python make_video.py

In [ ]:
# @title Watch it
import glob
from IPython.display import Video
mp4 = sorted(glob.glob("/content/runs/colab/*/videos/*.mp4"))[-1]
Video(mp4, embed=True, width=720)

In [ ]:
# @title Run report (aggregates every eval_record.json under runs/)
from deepracer_genesis.experiment.report import build_report
from IPython.display import Markdown
build_report("/content/runs", out_md="/content/runs/report.md")
Markdown(open("/content/runs/report.md").read())

## Where to go from here

- **Variants**: subclass `ColabRacer` and override attributes
  (`class Longer(ColabRacer): total_env_steps = 10_000_000`) — each variant
  gets its own content-addressed run directory.
- **Ablations**: `from deepracer_genesis.experiment.ablation import sweep, seeds`.
- **Camera policies + domain randomization** (`CameraEnvironment`,
  `DomainRandomizationCamera`, ...) need the Madrona/Nyx renderers — run those
  on a machine with NVIDIA Vulkan (not Colab).
- **Custom algorithms** (SAC, world models): implement the `Algorithm`
  protocol in `deepracer_genesis/experiment/algorithms.py`, register with
  `@register_algorithm("my_kind")`, select via `>> Algo(kind="my_kind")`.